In [73]:
import warnings
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy import stats
from numpy.linalg import LinAlgError

warnings.filterwarnings('ignore')
%matplotlib inline
%load_ext autoreload
%autoreload 2

pd.options.display.float_format = "{:,.3f}".format
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


INFILE_SHEET = '/home/grace/work/PPCG_DifferentialGenesetMutation/samplesheet.angel.alldonors.tsv'
INFILE_TISSUES = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/Sample_Donor_Tissue_Origin_01_June_2020.csv'
INFILE_MUTATIONS = '/home/grace/work/PPCG_DifferentialGenesetMutation/outputs/angel_mutations_all_donors/variant_processing/mutations.filtered.tsv'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
COMBI_TREAT_DONORS = [
    'PPCG0086', 'PPCG0087', 'PPCG0179', 'PPCG0180', 'PPCG0181', 'PPCG0387', 'PPCG0388', 'PPCG0389', 'PPCG0393', 'PPCG0394', 'PPCG0399', 'PPCG0404', 'PPCG0405', 'PPCG0415', 'PPCG0421', 'PPCG0423', 'PPCG0431', 'PPCG0432', 'PPCG0435', 'PPCG0467', 'PPCG0988', 'PPCG0990', 'PPCG0992', 'PPCG0994', 'PPCG0996', 'PPCG0998', 'PPCG1000', 'PPCG1002', 'PPCG1004', 'PPCG1006', 'PPCG1008', 'PPCG1010', 'PPCG1012', 'PPCG1014', 'PPCG1016', 'PPCG1018', 'PPCG1020', 'PPCG1022', 'PPCG1024', 'PPCG1026', 'PPCG1028', 'PPCG1030', 'PPCG1032', 'PPCG1034', 'PPCG1036', 'PPCG1038', 'PPCG1040', 'PPCG1042', 'PPCG1044', 'PPCG1048', 'PPCG1050', 'PPCG1052', 'PPCG1054', 'PPCG1056', 'PPCG1060', 'PPCG1066', 'PPCG1068', 'PPCG1070', 'PPCG1074', 'PPCG1076', 'PPCG1078', 'PPCG1080', 'PPCG1082'
    # removed 'PPCG0088', 'PPCG0089', 'PPCG0182', 'PPCG0183', 'PPCG0390'
]
PPCG_CTRL_DONORS = [
    'PPCG0001', 'PPCG0003', 'PPCG0004', 'PPCG0006', 'PPCG0008', 'PPCG0010', 'PPCG0017', 'PPCG0019', 'PPCG0022', 'PPCG0052', 'PPCG0053', 'PPCG0074', 'PPCG0134', 'PPCG0138', 'PPCG0159', 'PPCG0161', 'PPCG0172', 'PPCG0174', 'PPCG0203', 'PPCG0261', 'PPCG0264', 'PPCG0275', 'PPCG0280', 'PPCG0283', 'PPCG0284', 'PPCG0289', 'PPCG0306', 'PPCG0307', 'PPCG0342', 'PPCG0347', 'PPCG0348', 'PPCG0349', 'PPCG0408', 'PPCG0443', 'PPCG0452', 'PPCG0487', 'PPCG0516', 'PPCG0517', 'PPCG0519', 'PPCG0553', 'PPCG0567', 'PPCG0568', 'PPCG0592', 'PPCG0594', 'PPCG0595', 'PPCG0612', 'PPCG0615', 'PPCG0631', 'PPCG0640', 'PPCG0696', 'PPCG0775', 'PPCG0792', 'PPCG0805', 'PPCG0813', 'PPCG0830', 'PPCG0831', 'PPCG0885', 'PPCG0889', 'PPCG0910', 'PPCG0963', 'PPCG0977'
    # removed 'PPCG0314', 'PPCG0525', 'PPCG0548', 'PPCG0837'
]
print(len(COMBI_TREAT_DONORS))
print(len(PPCG_CTRL_DONORS))

63
61


### Binary Outcome

load data

In [75]:
# load mutations
muts = pd.read_csv(INFILE_MUTATIONS, sep='\t', header=0)

# filter donors
valid = set(COMBI_TREAT_DONORS) | set(PPCG_CTRL_DONORS)
muts = muts[muts['donor'].isin(valid)].copy()

# annotate binary outcome (cohort)
mask1 = muts['donor'].isin(set(COMBI_TREAT_DONORS))
mask2 = muts['donor'].isin(set(PPCG_CTRL_DONORS))
muts.loc[mask1, 'cohort'] = 'COMBI'
muts.loc[mask2, 'cohort'] = 'PPCG'

# annotate tissue
tis = pd.read_csv(INFILE_TISSUES, sep=',', header=0)
tis['sample'] = tis['PPCG_Sample_ID'].apply(lambda x: x[:9])
tmap = tis.set_index('sample')['Tissue_Origin'].to_dict()
muts['tissue'] = muts['sample'].map(tmap)

muts.head()


,sample,donor,coords,ref,alt,vclass,vtype,annotation,est_ccf,gene,cohort,tissue
575,PPCG0813a,PPCG0813,8:83622614-10:89633013,.,.,SV,TRA,transcript_ablation,NaN,PTEN,PPCG,Primary
576,PPCG0813a,PPCG0813,10:82168318-10:89657559,.,.,SV,INV,bidirectional_gene_fusion,NaN,PRXL2A,PPCG,Primary
577,PPCG0813a,PPCG0813,10:82168318-10:89657559,.,.,SV,INV,bidirectional_gene_fusion,NaN,PTEN,PPCG,Primary
578,PPCG0813a,PPCG0813,1:11311920-17:47301876,.,.,SV,TRA,gene_fusion&frameshift_variant,NaN,MTOR,PPCG,Primary
579,PPCG0813a,PPCG0813,1:11311920-17:47301876,.,.,SV,TRA,gene_fusion&frameshift_variant,NaN,PHOSPHO1,PPCG,Primary


sanity check

In [ ]:
# ensure combi donors each have primary tissue & met tissue
counts = muts[muts['cohort']=='COMBI'].groupby('donor')['tissue'].agg(set).to_frame().reset_index()
counts['has_primary'] = counts['tissue'].apply(lambda x: 'Primary' in x)
counts['has_metastasis'] = counts['tissue'].apply(lambda x: 'Metastasis' in x)
assert counts['has_primary'].all()
assert counts['has_metastasis'].all()

# ensure combi donors have 2+ samples 
counts = muts[muts['cohort']=='COMBI'].groupby('donor')['sample'].agg(set).to_frame().reset_index()
counts['n_samples'] = counts['sample'].apply(lambda x: len(x))
assert counts['n_samples'].min() >= 2

# ensure ppcg donors have 1 sample (and it's primary)
counts = muts[muts['cohort']=='PPCG'].groupby('donor')['sample'].agg(set).to_frame().reset_index()
counts['n_samples'] = counts['sample'].apply(lambda x: len(x))
assert counts['n_samples'].max() == 1
assert counts['n_samples'].min() == 1

print(muts.groupby('cohort')['tissue'].agg(set))
muts.head()

cohort
COMBI    {Primary, Recurrence, Metastasis}
PPCG                             {Primary}
Name: tissue, dtype: object


,sample,donor,coords,ref,alt,vclass,vtype,annotation,est_ccf,gene,cohort,tissue
575,PPCG0813a,PPCG0813,8:83622614-10:89633013,.,.,SV,TRA,transcript_ablation,NaN,PTEN,PPCG,Primary
576,PPCG0813a,PPCG0813,10:82168318-10:89657559,.,.,SV,INV,bidirectional_gene_fusion,NaN,PRXL2A,PPCG,Primary
577,PPCG0813a,PPCG0813,10:82168318-10:89657559,.,.,SV,INV,bidirectional_gene_fusion,NaN,PTEN,PPCG,Primary
578,PPCG0813a,PPCG0813,1:11311920-17:47301876,.,.,SV,TRA,gene_fusion&frameshift_variant,NaN,MTOR,PPCG,Primary
579,PPCG0813a,PPCG0813,1:11311920-17:47301876,.,.,SV,TRA,gene_fusion&frameshift_variant,NaN,PHOSPHO1,PPCG,Primary


Remove duplicates (each gene can only be hit )

Mark PPCG mutations as 'primary'

In [ ]:
muts['cig'] = muts['donor'] + '' + muts['']